# Clase 4 — Criptografía Simétrica de Flujo y MACs

> **Curso:** Criptografía Aplicada
> **Objetivo:** Comprender el diseño, funcionamiento interno e implementación práctica de los cifrados de flujo (RC4, Salsa20/ChaCha20) y los algoritmos de integridad de mensajes (HMAC, CMAC), culminando con los esquemas AEAD que combinan confidencialidad e integridad.

---

## Tabla de Contenidos

1. [Introducción a los cifrados de flujo](#1)
2. [Generadores de flujo de clave (CSPRNG)](#2)
   - 2.1 [LFSRs y sus limitaciones](#2-1)
   - 2.2 [Propiedades de un CSPRNG](#2-2)
3. [RC4](#3)
   - 3.1 [Algoritmo KSA y PRGA](#3-1)
   - 3.2 [Implementación en Python](#3-2)
   - 3.3 [Vulnerabilidades conocidas](#3-3)
4. [Salsa20 y ChaCha20](#4)
   - 4.1 [Estructura quarterround](#4-1)
   - 4.2 [Implementación de ChaCha20 en Python (RFC 7539)](#4-2)
   - 4.3 [ChaCha20 con la librería `cryptography`](#4-3)
   - 4.4 [Comparación de avalancha: RC4 vs ChaCha20](#4-4)
5. [Integridad de mensajes y MACs](#5)
   - 5.1 [Funciones hash vs MACs](#5-1)
   - 5.2 [HMAC — Keyed-Hash MAC](#5-2)
   - 5.3 [CMAC — Cipher-based MAC](#5-3)
   - 5.4 [Poly1305](#5-4)
   - 5.5 [Benchmark de MACs](#5-5)
6. [Cifrado Autenticado (AEAD)](#6)
   - 6.1 [AES-GCM](#6-1)
   - 6.2 [ChaCha20-Poly1305](#6-2)
   - 6.3 [Encrypt-then-MAC vs AEAD](#6-3)
7. [Ataques sobre MACs](#7)
   - 7.1 [Ataque de extensión de longitud](#7-1)
   - 7.2 [Brute-force de clave débil](#7-2)
8. [Rendimiento comparativo](#8)
9. [Ejercicios propuestos](#9)
10. [Referencias](#10)

---


## Importaciones globales

In [ ]:
# Librería estándar
import os, struct, time, hashlib, hmac as _hmac_std

# cryptography (pip install cryptography)
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes as _modes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM, ChaCha20Poly1305
from cryptography.hazmat.primitives import hashes, cmac as _cmac_module
from cryptography.hazmat.primitives.hmac import HMAC as CryptoHMAC
from cryptography.hazmat.primitives.poly1305 import Poly1305
from cryptography.hazmat.backends import default_backend
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110

print("Entorno listo ✓")


---
<a id='1'></a>
## 1. Introducción a los cifrados de flujo

Un **cifrado de flujo** (*stream cipher*) cifra la información byte a byte generando un flujo de clave (*keystream*) que se combina con el texto plano mediante XOR:

```
c_i = p_i ⊕ k_i        (cifrado)
p_i = c_i ⊕ k_i        (descifrado — idéntico)
```

La seguridad depende **enteramente** de que el keystream sea indistinguible de una secuencia aleatoria uniforme.

### Comparación con cifrados de bloque

| Característica          | Cifrado de bloque | Cifrado de flujo  |
|-------------------------|-------------------|-------------------|
| Unidad de proceso       | Bloque (64–128 b) | Byte / bit        |
| Estado interno          | Sin estado propio | Estado evoluciona |
| Paralelismo nativo      | Solo en CTR/GCM   | Limitado (sínc.)  |
| Latencia                | Mayor             | Menor             |
| Reuso de IV             | Peligroso         | **Catastrófico**  |
| Ejemplos modernos       | AES               | ChaCha20          |

> ⚠️ **Regla de oro**: nunca reutilice el par `(clave, nonce/IV)`.
> Si el atacante obtiene `c₁ ⊕ c₂ = p₁ ⊕ p₂`, ambos mensajes quedan expuestos (*two-time pad*).

### Taxonomía

```
Cifrado de flujo
├── Síncrono:  k_i = f(K, IV, i)   — independiente del plaintext/ciphertext
│   └── RC4, ChaCha20, CTR-AES
└── Asíncrono: k_i = f(K, prev_c)  — auto-sincronizante
    └── CFB, OFB (modos de AES)
```


---
<a id='2'></a>
## 2. Generadores de flujo de clave (CSPRNG)

El núcleo de un cifrado de flujo es su **generador pseudoaleatorio criptográfico** (CSPRNG).

<a id='2-1'></a>
### 2.1 LFSRs y sus limitaciones

Los **Linear Feedback Shift Registers (LFSR)** fueron la base de cifrados hardware clásicos (A5/1 en GSM, E0 en Bluetooth).

```
Estado:  [b_{n-1}, ..., b_1, b_0]
Salida:  b_0
Nuevo:   b_{n-1} = XOR de bits seleccionados por polinomio primitivo
```

**Vulnerabilidad:** el algoritmo de Berlekamp-Massey reconstruye el estado completo con solo `2n` bits del keystream. Los LFSRs nunca se usan solos.


In [ ]:
# LFSR ilustrativo (NO criptográfico)
def lfsr(seed: int, taps: list, n_bits: int) -> list:
    """LFSR de longitud max(taps) bits."""
    length = max(taps)
    mask   = (1 << length) - 1
    state  = seed & mask
    output = []
    for _ in range(n_bits):
        bit = state & 1
        output.append(bit)
        fb = 0
        for t in taps:
            fb ^= (state >> (t - 1)) & 1
        state = ((state >> 1) | (fb << (length - 1))) & mask
    return output

# LFSR de 16 bits — polinomio primitivo x^16+x^15+x^13+x^4+1
seed16 = 0xACE1
bits   = lfsr(seed16, [16, 15, 13, 4], 128)
print("Primeros 128 bits:", "".join(map(str, bits)))
print(f"Período máximo teórico: {2**16 - 1:,}")

# Distribución de los primeros 512 bytes
vals = [int("".join(map(str, bits[i:i+8])), 2) for i in range(0, len(bits)-7, 8)]
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(vals, bins=16, color="steelblue", alpha=0.8)
ax.set_title("Distribución de bytes del LFSR-16 (semilla=0xACE1)")
ax.set_xlabel("Valor byte")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()


<a id='2-2'></a>
### 2.2 Propiedades de un CSPRNG

| Propiedad | Descripción |
|-----------|-------------|
| **Next-bit unpredictability** | Conocer los primeros `k` bits no permite predecir el bit `k+1` con prob. > 1/2 |
| **State compromise extension** | Comprometer el estado actual no revela salidas pasadas |
| **Período exponencial** | ≥ 2^128 en la práctica |
| **Pasa tests estadísticos** | NIST SP 800-22, TestU01, Diehard |

Los CSPRNGs modernos (ChaCha20, AES-CTR) se basan en primitivas simétricas con pruebas de seguridad reduccionistas.


---
<a id='3'></a>
## 3. RC4

RC4 (*Rivest Cipher 4* / **ARC4**) fue diseñado por Ron Rivest en 1987, secreto hasta 1994. Fue el cifrado de flujo más usado durante décadas: WEP, WPA-TKIP, SSL/TLS (hoy prohibido).

<a id='3-1'></a>
### 3.1 Algoritmos KSA y PRGA

**KSA — Key Scheduling Algorithm** (inicializa la S-Box de 256 bytes):
```
S = [0, 1, ..., 255]
j = 0
for i in range(256):
    j = (j + S[i] + key[i % len(key)]) % 256
    swap(S[i], S[j])
```

**PRGA — Pseudo-Random Generation Algorithm** (genera el keystream):
```
i = j = 0
while True:
    i = (i + 1) % 256
    j = (j + S[i]) % 256
    swap(S[i], S[j])
    yield S[(S[i] + S[j]) % 256]
```


In [ ]:
# RC4 — Implementación educativa (NO usar en producción)
class RC4:
    """RC4/ARC4 desde cero. Solo para estudio."""

    def __init__(self, key: bytes):
        assert 1 <= len(key) <= 256, "Clave: 1-256 bytes"
        self.S = self._ksa(key)

    @staticmethod
    def _ksa(key: bytes) -> list:
        S = list(range(256))
        j = 0
        for i in range(256):
            j = (j + S[i] + key[i % len(key)]) % 256
            S[i], S[j] = S[j], S[i]
        return S

    def keystream(self):
        """Generador infinito del keystream."""
        S = self.S[:]
        i = j = 0
        while True:
            i = (i + 1) % 256
            j = (j + S[i]) % 256
            S[i], S[j] = S[j], S[i]
            yield S[(S[i] + S[j]) % 256]

    def crypt(self, data: bytes) -> bytes:
        ks = self.keystream()
        return bytes(b ^ next(ks) for b in data)


# Demo básica
key = b"ClaveRC4Demo"
msg = b"Hola, criptografía de flujo!"
ct  = RC4(key).crypt(msg)
pt  = RC4(key).crypt(ct)
print(f"Original  : {msg}")
print(f"Cifrado   : {ct.hex()}")
print(f"Descifrado: {pt}")
assert pt == msg, "Error de descifrado"
print("Verificación OK ✓")


In [ ]:
# Visualización del keystream de RC4
N = 1024
ks_a = [next(RC4(b"keyA").keystream()) for _ in range(N)]
ks_b = [next(RC4(b"keyB").keystream()) for _ in range(N)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ks_a[:256], linewidth=0.7, color="steelblue", label="keyA")
axes[0].plot(ks_b[:256], linewidth=0.7, color="tomato", alpha=0.7, label="keyB")
axes[0].set_title("Primeros 256 bytes del keystream RC4")
axes[0].set_xlabel("Posición")
axes[0].set_ylabel("Valor (0–255)")
axes[0].legend()

axes[1].hist(ks_a, bins=32, color="steelblue", alpha=0.75, label="keyA")
axes[1].hist(ks_b, bins=32, color="tomato", alpha=0.55, label="keyB")
axes[1].axhline(N/32, linestyle="--", color="gray", label="Esperado uniforme")
axes[1].set_title("Distribución de valores (N=1024)")
axes[1].set_xlabel("Valor byte")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Media ks_a: {sum(ks_a)/N:.2f}  (esperado ≈ 127.5)")


<a id='3-3'></a>
### 3.3 Vulnerabilidades conocidas de RC4

| Vulnerabilidad | Descripción | Consecuencia |
|----------------|-------------|--------------|
| **Sesgo en byte[1]** | El segundo byte del keystream tiene distribución no uniforme | Recuperación estadística de bytes del plaintext |
| **WEP flaw** (2001, FMS) | IV de 24 bits concatenado a la clave; permite Key Recovery con ~1M paquetes | Ruptura total de WEP en minutos |
| **BEAST/Lucky13** (2013) | RC4 en TLS 1.0/1.1 + peticiones repetidas revelan cookies | Ataque estadístico en ~2^26 peticiones |
| **Bar-Mitzvah** (2015) | LSSs en el estado inicial → sesgo en bits del keystream | MITM en TLS |
| **RFC 7465** (2015) | **Prohíbe** RC4 en TLS definitivamente | — |

> 🚫 **RC4 está obsoleto.** No debe usarse en ningún sistema nuevo.


In [ ]:
# Demostración del sesgo en el byte[1] del keystream RC4
import random

SAMPLES = 100_000
KEY_LEN = 16
byte1_counts = [0] * 256

for _ in range(SAMPLES):
    key = bytes(random.randint(0, 255) for _ in range(KEY_LEN))
    ks  = RC4(key).keystream()
    next(ks)               # byte[0]
    byte1_counts[next(ks)] += 1  # byte[1]

expected = SAMPLES / 256
bias_pct = [(c - expected) / expected * 100 for c in byte1_counts]

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(256), bias_pct, width=1.0, color="tomato", alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Sesgo relativo del byte[1] del keystream RC4 ({SAMPLES:,} claves)")
ax.set_xlabel("Valor del byte")
ax.set_ylabel("Desviación respecto al esperado (%)")
plt.tight_layout()
plt.show()

top = sorted(enumerate(byte1_counts), key=lambda x: -x[1])[:5]
print("Top 5 bytes más frecuentes en posición [1]:")
for val, cnt in top:
    print(f"  0x{val:02X} → {cnt} veces (sesgo {(cnt-expected)/expected*100:+.2f}%)")


---
<a id='4'></a>
## 4. Salsa20 y ChaCha20

**Salsa20** (Bernstein, 2005) y su variante **ChaCha20** (2008) son los cifrados de flujo más seguros y rápidos en software. ChaCha20 es el estándar de facto en TLS 1.3, WireGuard, Signal y OpenSSH.

### Estado interno (512 bits = 16 palabras de 32 bits)

```
+────────────+────────────+────────────+────────────+
│  "expa"    │  "nd 3"    │  "2-by"    │  "te k"    │  ← 4 constantes
+────────────+────────────+────────────+────────────+
│  key[ 0]   │  key[ 1]   │  key[ 2]   │  key[ 3]   │
│  key[ 4]   │  key[ 5]   │  key[ 6]   │  key[ 7]   │  ← clave 256 bits
+────────────+────────────+────────────+────────────+
│  counter   │ nonce[ 0]  │ nonce[ 1]  │ nonce[ 2]  │  ← 32+96 bits
+────────────+────────────+────────────+────────────+
```

<a id='4-1'></a>
### 4.1 Estructura quarterround

Mezcla 4 palabras de 32 bits con **Add-Rotate-XOR (ARX)**:
```
a += b;  d ^= a;  d <<<= 16
c += d;  b ^= c;  b <<<= 12
a += b;  d ^= a;  d <<<= 8
c += d;  b ^= c;  b <<<= 7
```
ChaCha20 aplica 10 double-rounds (20 rondas): primero en columnas, luego en diagonales.


In [ ]:
# ChaCha20 — Implementación completa (RFC 7539)
import struct

CHACHA_CONST = b"expand 32-byte k"

def rotl32(v: int, n: int) -> int:
    return ((v << n) | (v >> (32 - n))) & 0xFFFFFFFF

def quarterround(s: list, a: int, b: int, c: int, d: int) -> None:
    s[a] = (s[a] + s[b]) & 0xFFFFFFFF; s[d] ^= s[a]; s[d] = rotl32(s[d], 16)
    s[c] = (s[c] + s[d]) & 0xFFFFFFFF; s[b] ^= s[c]; s[b] = rotl32(s[b], 12)
    s[a] = (s[a] + s[b]) & 0xFFFFFFFF; s[d] ^= s[a]; s[d] = rotl32(s[d],  8)
    s[c] = (s[c] + s[d]) & 0xFFFFFFFF; s[b] ^= s[c]; s[b] = rotl32(s[b],  7)

def chacha20_block(key: bytes, counter: int, nonce: bytes) -> bytes:
    """Genera 64 bytes del keystream para el bloque `counter`."""
    assert len(key) == 32 and len(nonce) == 12
    const   = struct.unpack("<4I", CHACHA_CONST)
    k       = struct.unpack("<8I", key)
    n       = struct.unpack("<3I", nonce)
    state   = list(const) + list(k) + [counter] + list(n)
    working = state[:]
    for _ in range(10):            # 10 double-rounds
        quarterround(working, 0, 4,  8, 12)
        quarterround(working, 1, 5,  9, 13)
        quarterround(working, 2, 6, 10, 14)
        quarterround(working, 3, 7, 11, 15)
        quarterround(working, 0, 5, 10, 15)
        quarterround(working, 1, 6, 11, 12)
        quarterround(working, 2, 7,  8, 13)
        quarterround(working, 3, 4,  9, 14)
    output = [(working[i] + state[i]) & 0xFFFFFFFF for i in range(16)]
    return struct.pack("<16I", *output)

def chacha20_encrypt(key: bytes, nonce: bytes, data: bytes, counter: int = 0) -> bytes:
    out = bytearray()
    for blk in range(0, len(data), 64):
        chunk  = data[blk: blk + 64]
        ks_blk = chacha20_block(key, counter + blk // 64, nonce)
        out.extend(b ^ k for b, k in zip(chunk, ks_blk))
    return bytes(out)


# Vector de prueba RFC 7539 §2.4.2
key_tv   = bytes.fromhex("000102030405060708090a0b0c0d0e0f"
                          "101112131415161718191a1b1c1d1e1f")
nonce_tv = bytes.fromhex("000000000000004a00000000")
msg_tv   = (b"Ladies and Gentlemen of the class of '99: "
            b"If I could offer you only one tip for the future, "
            b"sunscreen would be it.")

ct_tv  = chacha20_encrypt(key_tv, nonce_tv, msg_tv, counter=1)
pt_tv  = chacha20_encrypt(key_tv, nonce_tv, ct_tv,  counter=1)
assert pt_tv == msg_tv
print("Vector RFC 7539 verificado ✓")
print("CT (hex):", ct_tv.hex()[:64], "...")


<a id='4-3'></a>
### 4.3 ChaCha20 con la librería `cryptography`

En producción se usan librerías auditadas; nunca implementaciones propias:


In [ ]:
# ChaCha20 con cryptography (solo cifrado, sin autenticación)
key_prod   = os.urandom(32)
nonce_prod = os.urandom(16)   # la librería usa 128 bits de nonce internamente
msg_prod   = b"Mensaje secreto de producción — versión de librería."

enc = Cipher(algorithms.ChaCha20(key_prod, nonce_prod), mode=None,
             backend=default_backend()).encryptor()
dec = Cipher(algorithms.ChaCha20(key_prod, nonce_prod), mode=None,
             backend=default_backend()).decryptor()

ct_prod = enc.update(msg_prod) + enc.finalize()
pt_prod = dec.update(ct_prod)  + dec.finalize()

print(f"CT: {ct_prod.hex()}")
print(f"PT: {pt_prod}")
assert pt_prod == msg_prod
print("OK ✓  — Para cifrado autenticado ver sección 6.2 (ChaCha20-Poly1305)")


<a id='4-4'></a>
### 4.4 Comparación de avalancha: RC4 vs ChaCha20

El **efecto avalancha** mide qué porcentaje de bits del keystream cambian al modificar un solo bit de la clave. Un buen cifrado apunta al ~50%.


In [ ]:
# Efecto avalancha: RC4 vs ChaCha20
def hamming_distance(a: bytes, b: bytes) -> int:
    return sum(bin(x ^ y).count("1") for x, y in zip(a, b))

def flip_bit(data: bytes, bit_pos: int) -> bytes:
    ba = bytearray(data)
    ba[bit_pos // 8] ^= 1 << (bit_pos % 8)
    return bytes(ba)

NBYTES = 64
KEY    = os.urandom(32)
NONCE  = os.urandom(12)
N_BITS = 128

rc4_avl   = []
cha_avl   = []

ks_rc4_base = bytes(next(RC4(KEY[:16]).keystream()) for _ in range(NBYTES))
ks_cha_base = chacha20_block(KEY, 0, NONCE)

for bit in range(N_BITS):
    kf = flip_bit(KEY, bit)
    ks_rc4_f = bytes(next(RC4(kf[:16]).keystream()) for _ in range(NBYTES))
    ks_cha_f = chacha20_block(kf, 0, NONCE)
    rc4_avl.append(hamming_distance(ks_rc4_base, ks_rc4_f) / (NBYTES * 8) * 100)
    cha_avl.append(hamming_distance(ks_cha_base, ks_cha_f) / (NBYTES * 8) * 100)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rc4_avl,  label="RC4",      color="tomato",    linewidth=1.2, alpha=0.85)
ax.plot(cha_avl,  label="ChaCha20", color="steelblue", linewidth=1.2)
ax.axhline(50, linestyle="--", color="gray", linewidth=0.8, label="Ideal 50 %")
ax.set_title("Efecto avalancha — cambio de 1 bit en la clave (primeros 128 bits)")
ax.set_xlabel("Bit de la clave modificado")
ax.set_ylabel("Bits del keystream modificados (%)")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Media RC4:      {sum(rc4_avl)/N_BITS:.1f} %")
print(f"Media ChaCha20: {sum(cha_avl)/N_BITS:.1f} %")


---
<a id='5'></a>
## 5. Integridad de mensajes y MACs

Un **Message Authentication Code (MAC)** es una etiqueta generada a partir de un mensaje `m` y una clave secreta `k`:
```
tag = MAC(k, m)
```

Garantiza **integridad** (el mensaje no fue alterado) y **autenticación de origen** (solo quien tiene `k` puede generar una etiqueta válida).

> Un MAC **no cifra** el mensaje. Para confidencialidad se combina con un cifrador → **AEAD** (sección 6).

<a id='5-1'></a>
### 5.1 Funciones hash vs MACs

| | Hash (SHA-256) | MAC (HMAC) |
|---|---|---|
| Clave secreta | No | Sí |
| Autenticación de origen | No | Sí |
| `H(secret ‖ m)` | ❌ Vulnerable a extensión de longitud | — |
| Velocidad | Muy alta | Alta (overhead mínimo) |


<a id='5-2'></a>
### 5.2 HMAC — Keyed-Hash Message Authentication Code

Definido en **RFC 2104** (1996). La doble aplicación previene el ataque de extensión de longitud:

```
HMAC(K, m) = H( (K ⊕ opad) ‖ H( (K ⊕ ipad) ‖ m ) )

ipad = 0x36 repetido (tamaño del bloque hash)
opad = 0x5C repetido (tamaño del bloque hash)
```

**Seguridad:** HMAC es una PRF segura si la función de compresión interna de H es una PRF segura (resultado más fuerte que collision-resistance de H).


In [ ]:
# HMAC-SHA256 implementado desde cero (RFC 2104)
import hmac as stdlib_hmac

def hmac_sha256(key: bytes, message: bytes) -> bytes:
    """Implementación educativa de HMAC-SHA256."""
    BLOCK = 64   # tamaño bloque SHA-256 en bytes
    OPAD  = bytes([0x5C] * BLOCK)
    IPAD  = bytes([0x36] * BLOCK)
    if len(key) > BLOCK:
        key = hashlib.sha256(key).digest()
    key = key.ljust(BLOCK, b"\x00")
    inner = hashlib.sha256(bytes(k ^ i for k, i in zip(key, IPAD)) + message).digest()
    outer = hashlib.sha256(bytes(k ^ o for k, o in zip(key, OPAD)) + inner).digest()
    return outer

# Verificación contra la librería estándar
key_h = b"clave_hmac_segura_256bits_12345"
msg_h = b"Transferencia: 1000 EUR a cuenta 987654"

tag_manual = hmac_sha256(key_h, msg_h)
tag_stdlib = stdlib_hmac.new(key_h, msg_h, hashlib.sha256).digest()

print(f"HMAC manual : {tag_manual.hex()}")
print(f"HMAC stdlib : {tag_stdlib.hex()}")
assert tag_manual == tag_stdlib
print("Verificación OK ✓")

# La verificación DEBE ser en tiempo constante para evitar timing attacks
valid = stdlib_hmac.compare_digest(tag_manual, tag_stdlib)
print(f"compare_digest (tiempo constante): {valid}")


In [ ]:
# HMAC con la librería cryptography — firma y verificación
def sign_msg(key: bytes, msg: bytes) -> bytes:
    h = CryptoHMAC(key, hashes.SHA256(), backend=default_backend())
    h.update(msg)
    return h.finalize()

def verify_msg(key: bytes, msg: bytes, tag: bytes) -> bool:
    h = CryptoHMAC(key, hashes.SHA256(), backend=default_backend())
    h.update(msg)
    try:
        h.verify(tag)
        return True
    except Exception:
        return False

key_v = os.urandom(32)
msg_v = b"Operacion: ACTIVAR SISTEMA a las 08:00"
tag_v = sign_msg(key_v, msg_v)

print(f"Mensaje  : {msg_v.decode()}")
print(f"Tag      : {tag_v.hex()}")
print(f"Verif. OK        : {verify_msg(key_v, msg_v, tag_v)}")
print(f"Verif. manipulado: {verify_msg(key_v, b'DESACTIVAR SISTEMA', tag_v)}")


<a id='5-3'></a>
### 5.3 CMAC — Cipher-based MAC

**CMAC** (NIST SP 800-38B) usa AES en lugar de SHA como primitiva. Ideal para entornos con aceleración hardware AES:

```
K1, K2 = derive_subkeys(AES_K(0^128))
CMAC = AES_K( AES_K(... AES_K(m₁) ⊕ m₂ ...) ⊕ mₙ* ⊕ Kᵢ )
```

- Seguridad: ≈ `n·2^(-b)` donde `b=128` bits (bloque AES).
- Preferido en IoT, HSMs, aplicaciones FIPS 140-2.


In [ ]:
# CMAC con la librería cryptography
from cryptography.hazmat.primitives import cmac as _cmac_mod
from cryptography.hazmat.primitives.ciphers import algorithms as _alg

def cmac_aes128(key: bytes, message: bytes) -> bytes:
    c = _cmac_mod.CMAC(_alg.AES(key), backend=default_backend())
    c.update(message)
    return c.finalize()

def cmac_verify(key: bytes, message: bytes, tag: bytes) -> bool:
    c = _cmac_mod.CMAC(_alg.AES(key), backend=default_backend())
    c.update(message)
    try:
        c.verify(tag)
        return True
    except Exception:
        return False

key_c = os.urandom(16)   # AES-128
msg_c = b"Comando critico: DEPLOY v2.3.1"
tag_c = cmac_aes128(key_c, msg_c)

print(f"CMAC-AES128 : {tag_c.hex()}")
print(f"Verif. OK   : {cmac_verify(key_c, msg_c, tag_c)}")
print(f"Verif. alt  : {cmac_verify(key_c, b'DEPLOY v9.9.9', tag_c)}")


<a id='5-4'></a>
### 5.4 Poly1305

Diseñado por Bernstein (2005). MAC de alta velocidad basado en polinomios sobre GF(2¹³⁰−5):

```
tag = ( r·m₁ + r²·m₂ + ... + rⁿ·mₙ + s ) mod (2¹³⁰ − 5)
```

- Clave **one-time** de 32 bytes: 16B para `r` (clamped) + 16B para `s`.
- Debe usarse una clave nueva por mensaje → emparejado con ChaCha20 para generarla de forma determinista.
- 128 bits de seguridad. Inmune a timing attacks en implementaciones sin tablas de búsqueda.


In [ ]:
# Poly1305 con cryptography
key_poly = os.urandom(32)   # one-time key (en producción: derivada de ChaCha20)
msg_poly = b"Datos a autenticar con Poly1305 — one-time key"

p = Poly1305(key_poly)
p.update(msg_poly)
tag_poly = p.finalize()
print(f"Poly1305 tag : {tag_poly.hex()}")

# Verificar
p2 = Poly1305(key_poly)
p2.update(msg_poly)
try:
    p2.verify(tag_poly)
    print("Verificación Poly1305 OK ✓")
except Exception as e:
    print(f"Error: {e}")

# Con mensaje alterado
p3 = Poly1305(key_poly)
p3.update(b"Datos ALTERADOS")
try:
    p3.verify(tag_poly)
    print("ERROR: no debería pasar")
except Exception:
    print("Mensaje alterado correctamente rechazado ✓")


### 5.5 Comparativa de MACs

| MAC | Primitiva | Tag | Velocidad | Uso típico |
|-----|-----------|-----|-----------|------------|
| HMAC-SHA256 | SHA-256 | 32 B | Alta | TLS, JWT, APIs |
| HMAC-SHA512 | SHA-512 | 64 B | Alta | Archivos, backups |
| HMAC-SHA3-256 | SHA3-256 | 32 B | Media | Post-quantum |
| CMAC-AES128 | AES-128 | 16 B | Muy alta (HW) | IoT, HSMs, FIPS |
| Poly1305 | GF(2¹³⁰−5) | 16 B | Muy alta (SW) | TLS 1.3, WireGuard |


In [ ]:
# Benchmark comparativo de MACs (64 KB, 200 iteraciones)
import timeit

DATA_BM = os.urandom(64 * 1024)
K256    = os.urandom(32)
K128    = os.urandom(16)
REPS    = 200

def bench(name, fn, reps=REPS):
    t    = timeit.timeit(fn, number=reps)
    mbs  = len(DATA_BM) * reps / t / 1e6
    print(f"  {name:<22}: {mbs:7.1f} MB/s  ({t/reps*1000:.3f} ms/iter)")

print(f"=== Benchmark MACs (mensaje: {len(DATA_BM)//1024} KB) ===")
bench("HMAC-SHA256",    lambda: stdlib_hmac.new(K256, DATA_BM, hashlib.sha256).digest())
bench("HMAC-SHA512",    lambda: stdlib_hmac.new(K256, DATA_BM, hashlib.sha512).digest())
bench("HMAC-SHA3-256",  lambda: stdlib_hmac.new(K256, DATA_BM, hashlib.sha3_256).digest())
bench("CMAC-AES128",    lambda: cmac_aes128(K128, DATA_BM))
bench("Poly1305",       lambda: Poly1305.generate_tag(os.urandom(32), DATA_BM))


---
<a id='6'></a>
## 6. Cifrado Autenticado (AEAD)

**AEAD** (*Authenticated Encryption with Associated Data*) combina:
- **Confidencialidad**: el mensaje es cifrado.
- **Integridad + Autenticación**: un tag cubre el ciphertext y datos adicionales (*associated data*, AD, no cifrados).

```
(ciphertext, tag) ← AEAD_Enc(K, N, plaintext, AD)
plaintext/⊥       ← AEAD_Dec(K, N, ciphertext, tag, AD)   # ⊥ si tag inválido
```

### Patrones Encrypt+MAC: por qué AEAD es mejor

| Patrón | Seguridad | Complejidad |
|--------|-----------|-------------|
| **Encrypt-then-MAC** | ✅ Seguro (si se implementa bien) | Alta (2 claves, orden crítico) |
| **MAC-then-Encrypt** | ❌ Vulnerable a padding oracle | Alta |
| **Encrypt-and-MAC** | ❌ MAC sobre plaintext filtra información | Alta |
| **AEAD** | ✅ Seguro por diseño | Baja (una primitiva) |

> 📌 **Regla moderna**: usar siempre AEAD. No combinar cifrado y MAC manualmente.

<a id='6-1'></a>
### 6.1 AES-GCM

AES-GCM = AES-CTR + GHASH (en GF(2¹²⁸)):
```
CT      = AES_CTR(K, IV, PT)
H       = AES_K(0¹²⁸)
Tag     = GHASH(H, AD, CT) ⊕ AES_K(IV ‖ 0³²‖1)
```
- IV recomendado: **12 bytes** aleatorios por mensaje.
- Tag: 16 bytes (128 bits).
- **CRÍTICO**: Reutilizar (K, IV) → revela PT₁⊕PT₂ **y** permite forjar tags.


In [ ]:
# AES-GCM con cryptography
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def gcm_enc(key: bytes, pt: bytes, aad: bytes = b"") -> tuple:
    gcm   = AESGCM(key)
    nonce = os.urandom(12)
    ct    = gcm.encrypt(nonce, pt, aad)
    return nonce, ct

def gcm_dec(key: bytes, nonce: bytes, ct: bytes, aad: bytes = b"") -> bytes:
    return AESGCM(key).decrypt(nonce, ct, aad)

key_gcm = AESGCM.generate_key(bit_length=256)
pt_gcm  = b"Datos financieros ultra secretos."
aad_gcm = b'{"usuario":"admin","op":"pago"}'

nonce_gcm, ct_gcm = gcm_enc(key_gcm, pt_gcm, aad_gcm)
dec_gcm           = gcm_dec(key_gcm, nonce_gcm, ct_gcm, aad_gcm)

print(f"Nonce         : {nonce_gcm.hex()}")
print(f"CT + Tag      : {ct_gcm.hex()}")
print(f"Tag (16B)     : {ct_gcm[-16:].hex()}")
print(f"Descifrado    : {dec_gcm}")
assert dec_gcm == pt_gcm
print("AES-GCM OK ✓")

# AAD manipulado → fallo
try:
    gcm_dec(key_gcm, nonce_gcm, ct_gcm, b"aad_alterado")
except Exception as e:
    print(f"AAD manipulado rechazado ✓ ({type(e).__name__})")


#### Demostración: reutilizar nonce en AES-GCM

In [ ]:
# Reutilización de nonce en AES-GCM — ataque two-time pad
key_bad   = os.urandom(32)
nonce_bad = os.urandom(12)      # ⚠️ mismo nonce para dos mensajes

m1 = b"Mensaje A confidencial--0000000"
m2 = b"Mensaje B confidencial--1111111"

gcm_bad = AESGCM(key_bad)
ct1 = gcm_bad.encrypt(nonce_bad, m1, None)
ct2 = gcm_bad.encrypt(nonce_bad, m2, None)

# El keystream es idéntico → XOR ciphertexts = XOR plaintexts
xor_ct = bytes(a ^ b for a, b in zip(ct1[:-16], ct2[:-16]))
xor_pt = bytes(a ^ b for a, b in zip(m1, m2))

print(f"XOR ciphertexts : {xor_ct.hex()}")
print(f"XOR plaintexts  : {xor_pt.hex()}")
print(f"Iguales?         {xor_ct == xor_pt}")

# Si el atacante conoce m1, recupera m2
recovered_m2 = bytes(x ^ p for x, p in zip(xor_ct, m1))
print(f"m2 recuperado   : {recovered_m2}")
assert recovered_m2 == m2
print("¡m2 totalmente comprometido! ✗ — NUNCA reutilice el nonce.")


<a id='6-2'></a>
### 6.2 ChaCha20-Poly1305

Definido en **RFC 8439**. Alternativa de software puro a AES-GCM:

```
poly_key   = ChaCha20_block(K, nonce, counter=0)[0:32]
ciphertext = ChaCha20(K, nonce, counter=1, PT)
tag        = Poly1305(poly_key, pad(AAD) ‖ pad(CT) ‖ len(AAD)‖len(CT))
```

**Ventajas sobre AES-GCM:**
- No requiere instrucciones AES-NI — seguro en ARM, MIPS, RISC-V sin hardware.
- Inmune a timing attacks de tablas GF(2¹²⁸) (si se implementa correctamente).
- Mismo nivel de seguridad (256 bits clave, 128 bits tag).

Adoptado en: **TLS 1.3**, **WireGuard**, **Signal Protocol**, **OpenSSH ≥ 6.5**, **QUIC**.


In [ ]:
# ChaCha20-Poly1305 con cryptography
from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305

def cp_enc(key: bytes, pt: bytes, aad: bytes = b"") -> tuple:
    cc    = ChaCha20Poly1305(key)
    nonce = os.urandom(12)
    ct    = cc.encrypt(nonce, pt, aad)
    return nonce, ct

def cp_dec(key: bytes, nonce: bytes, ct: bytes, aad: bytes = b"") -> bytes:
    return ChaCha20Poly1305(key).decrypt(nonce, ct, aad)

key_cp = ChaCha20Poly1305.generate_key()
pt_cp  = b"Mensaje Signal privado — ChaCha20-Poly1305"
aad_cp = b"v1|alice|bob|seq=42"

nonce_cp, ct_cp = cp_enc(key_cp, pt_cp, aad_cp)
dec_cp          = cp_dec(key_cp, nonce_cp, ct_cp, aad_cp)

print(f"Nonce : {nonce_cp.hex()}")
print(f"CT+Tag: {ct_cp.hex()}")
print(f"Descif: {dec_cp}")
assert dec_cp == pt_cp
print("ChaCha20-Poly1305 OK ✓")


<a id='6-3'></a>
### 6.3 Encrypt-then-MAC correcto (cuando AEAD no está disponible)

Si se deben usar primitivas separadas, el orden correcto es **Encrypt-then-MAC** (EtM): el MAC se calcula sobre el ciphertext, no sobre el plaintext.


In [ ]:
# Encrypt-then-MAC: AES-256-CBC + HMAC-SHA256
def pkcs7_pad(data: bytes, bs: int = 16) -> bytes:
    n = bs - len(data) % bs
    return data + bytes([n] * n)

def pkcs7_unpad(data: bytes) -> bytes:
    n = data[-1]
    if n < 1 or n > 16:
        raise ValueError("Padding inválido")
    return data[:-n]

def etm_enc(enc_key: bytes, mac_key: bytes, pt: bytes, aad: bytes = b"") -> bytes:
    """Encrypt-then-MAC: AES-256-CBC + HMAC-SHA256."""
    iv  = os.urandom(16)
    enc = Cipher(algorithms.AES(enc_key), _modes.CBC(iv),
                 backend=default_backend()).encryptor()
    ct  = enc.update(pkcs7_pad(pt)) + enc.finalize()
    # MAC cubre AAD + IV + CT (previene ataques de reordenamiento y replay)
    mac = stdlib_hmac.new(mac_key, aad + iv + ct, hashlib.sha256).digest()
    return iv + ct + mac

def etm_dec(enc_key: bytes, mac_key: bytes, pkt: bytes, aad: bytes = b"") -> bytes:
    if len(pkt) < 16 + 16 + 32:
        raise ValueError("Paquete muy corto")
    iv, ct, mac = pkt[:16], pkt[16:-32], pkt[-32:]
    expected = stdlib_hmac.new(mac_key, aad + iv + ct, hashlib.sha256).digest()
    if not stdlib_hmac.compare_digest(mac, expected):
        raise ValueError("MAC inválido — posible manipulación")
    dec = Cipher(algorithms.AES(enc_key), _modes.CBC(iv),
                 backend=default_backend()).decryptor()
    return pkcs7_unpad(dec.update(ct) + dec.finalize())

enc_k = os.urandom(32)
mac_k = os.urandom(32)
msg   = b"Datos sensibles con EtM correcto"
pkt   = etm_enc(enc_k, mac_k, msg, aad=b"canal_seguro")
dec   = etm_dec(enc_k, mac_k, pkt, aad=b"canal_seguro")
assert dec == msg
print("Encrypt-then-MAC correcto ✓")

# Manipulación del ciphertext → MAC falla
try:
    bad_pkt = pkt[:20] + bytes([pkt[20] ^ 0xFF]) + pkt[21:]
    etm_dec(enc_k, mac_k, bad_pkt, aad=b"canal_seguro")
except ValueError as e:
    print(f"Manipulación detectada ✓: {e}")


---
<a id='7'></a>
## 7. Ataques sobre MACs

<a id='7-1'></a>
### 7.1 Ataque de extensión de longitud (*Length Extension Attack*)

Afecta a `H(secret ‖ message)` con funciones hash Merkle-Damgård (MD5, SHA-1, SHA-256, SHA-512).

El estado interno de SHA-256 después de procesar `secret ‖ msg` es precisamente el hash final. Un atacante puede continuar el hash desde ese estado para calcular `H(secret ‖ msg ‖ padding ‖ extension)` **sin conocer el secreto**:

```
H_extendido = SHA256_continue(H(secret‖msg), extension)
            = H(secret ‖ msg ‖ glue_padding ‖ extension)
```

**HMAC es inmune** gracias a la estructura doble `H(K⊕opad ‖ H(K⊕ipad ‖ m))`.


In [ ]:
# Demostración simplificada del ataque de extensión de longitud
# (la verificación usa el secreto para mostrar que el forjado es correcto)

def sha256_padding(msg_len: int) -> bytes:
    """Padding Merkle-Damgård para SHA-256 (mensaje de msg_len bytes)."""
    bit_len = msg_len * 8
    pad = b"\x80"
    pad += b"\x00" * ((55 - msg_len) % 64)
    pad += struct.pack(">Q", bit_len)
    return pad

# MAC vulnerable: SHA256(secret ‖ msg) — NO HACER ESTO
secret   = b"secreto_16bytes!"   # 16 bytes
msg_orig = b"user=alice&role=user"

vuln_mac = hashlib.sha256(secret + msg_orig).digest()
print(f"MAC vulnerable  : {vuln_mac.hex()}")

# El atacante conoce: msg_orig, len(secret)=16, y vuln_mac
# Quiere forjar: SHA256(secret ‖ msg_orig ‖ glue ‖ extension)
glue      = sha256_padding(len(secret) + len(msg_orig))
extension = b"&role=admin"
forged    = msg_orig + glue + extension

# Verificación: recalcular con el secreto (el atacante NO haría esto)
forged_mac = hashlib.sha256(secret + forged).digest()
print(f"MAC del forjado : {forged_mac.hex()}")
print(f"Msg forjado     : {forged!r}")
print()

# HMAC NO es vulnerable
hmac_mac = stdlib_hmac.new(secret, msg_orig, hashlib.sha256).digest()
# El estado interno de HMAC no permite extensión sin la clave
print(f"HMAC (seguro)   : {hmac_mac.hex()}")
print("Para extender HMAC necesitas la clave → imposible sin ella ✓")


<a id='7-2'></a>
### 7.2 Brute-force de clave débil

Si la clave del MAC tiene poca entropía (ej. PIN de 4 dígitos), puede recuperarse por fuerza bruta.


In [ ]:
# Brute-force de HMAC con clave = PIN de 4 dígitos (10^4 posibilidades)
import time as _time

secret_pin = b"4827"                       # clave real (atacante no la conoce)
msg_bf     = b"action=transfer&amount=500"
tag_bf     = stdlib_hmac.new(secret_pin, msg_bf, hashlib.sha256).digest()

print(f"Tag objetivo    : {tag_bf.hex()[:16]}...")
print(f"Espacio: 10^4 = {10**4:,} candidatos")

found = None
t0    = _time.perf_counter()
for pin in range(10000):
    candidate = f"{pin:04d}".encode()
    if stdlib_hmac.compare_digest(
        stdlib_hmac.new(candidate, msg_bf, hashlib.sha256).digest(), tag_bf
    ):
        found = candidate
        break
elapsed = _time.perf_counter() - t0

print(f"Clave encontrada: {found.decode()!r} en {elapsed*1000:.1f} ms")
print()
print("Lección: HMAC es seguro SOLO con claves de alta entropía.")
print("Generar con: key = os.urandom(32)  # 256 bits de entropía")


---
<a id='8'></a>
## 8. Rendimiento comparativo

Comparación de cifrados AEAD sobre distintos tamaños de mensaje.


In [ ]:
import timeit

SIZES   = [1024, 16*1024, 64*1024, 256*1024]
REPS_P  = 80

res = {"AES-GCM-256": [], "ChaCha20-Poly1305": [], "AES-CBC+HMAC (EtM)": []}

k_gcm2  = AESGCM.generate_key(bit_length=256)
k_cp2   = ChaCha20Poly1305.generate_key()
enc_k2  = os.urandom(32)
mac_k2  = os.urandom(32)
gcm2    = AESGCM(k_gcm2)
cp2     = ChaCha20Poly1305(k_cp2)
_nonce  = os.urandom(12)

for sz in SIZES:
    data = os.urandom(sz)
    t = timeit.timeit(lambda: gcm2.encrypt(_nonce, data, b"aad"), number=REPS_P)
    res["AES-GCM-256"].append(sz * REPS_P / t / 1e6)
    t = timeit.timeit(lambda: cp2.encrypt(_nonce, data, b"aad"), number=REPS_P)
    res["ChaCha20-Poly1305"].append(sz * REPS_P / t / 1e6)
    t = timeit.timeit(lambda: etm_enc(enc_k2, mac_k2, data), number=REPS_P)
    res["AES-CBC+HMAC (EtM)"].append(sz * REPS_P / t / 1e6)

labels = [f"{s//1024} KB" for s in SIZES]
fig, ax = plt.subplots(figsize=(10, 5))
for algo, vals in res.items():
    ax.plot(range(len(SIZES)), vals, marker="o", label=algo, linewidth=2)
ax.set_xticks(range(len(SIZES)))
ax.set_xticklabels(labels)
ax.set_xlabel("Tamaño del mensaje")
ax.set_ylabel("Rendimiento (MB/s)")
ax.set_title("Benchmark: cifrado AEAD por tamaño de mensaje")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

# Tabla resumen
print(f"\n{'Algoritmo':<25} {'1 KB':>8} {'16 KB':>8} {'64 KB':>8} {'256 KB':>9}")
print("-" * 60)
for algo, vals in res.items():
    row = "  ".join(f"{v:7.1f}" for v in vals)
    print(f"{algo:<25} {row}")
print("Unidades: MB/s")


---
<a id='9'></a>
## 9. Ejercicios propuestos

### Nivel básico

**E1. RC4-drop256:** Implemente RC4 descartando los primeros 256 bytes del keystream (*RC4-drop256*). Compare la distribución de sesgos del byte[1] con la versión sin drop. ¿Cuánto mejora?

**E2. Cifrar archivos con AEAD:** Cifre un archivo de texto con `ChaCha20-Poly1305` y guárdelo en disco con el formato `nonce(12B) ‖ ciphertext+tag`. Luego descífrelo verificando el tag.

**E3. HMAC básico:** Genere 3 tags `HMAC-SHA256` para el mensaje `"pago:100EUR"` con tres claves distintas. Compruebe que son distintos. ¿Qué ocurre si modifica un carácter del mensaje?

### Nivel intermedio

**E4. Two-time pad:** Cifre dos mensajes distintos con la misma clave y nonce en ChaCha20. Demuestre algebraicamente y empíricamente que el XOR de los ciphertexts revela información.

**E5. JWT simplificado:** Implemente un esquema de token `HEADER.PAYLOAD.SIG` donde `SIG = HMAC-SHA256(key, base64url(H)+"." + base64url(P))`. Verifique que un token alterado es rechazado.

**E6. CMAC vs HMAC:** Compare CMAC-AES128 y HMAC-SHA256 para mensajes de 16, 64, 256 y 4096 bytes. ¿En qué rango CMAC supera a HMAC? ¿Por qué?

### Nivel avanzado

**E7. Length extension attack completo:** Implemente el ataque de extensión de longitud sobre SHA-256 manipulando el estado interno con `struct` y el valor hash como estado inicial de la función de compresión (sin usar librerías externas como `hashpumpy`). Verifique el resultado.

**E8. AES-GCM-SIV:** Investigue la construcción AES-GCM-SIV (RFC 8452). ¿Cómo garantiza seguridad ante reutilización de nonce? Implemente un ejemplo con `cryptography` (requiere versión ≥ 41.0).

**E9. Protocolo de mensajería segura:** Diseñe e implemente un protocolo cliente-servidor que use:
- ECDH (X25519) para intercambio de claves.
- `ChaCha20-Poly1305` para AEAD con un contador de secuencia de 64 bits embebido en el nonce.
- `HMAC-SHA256` sobre `(timestamp ‖ public_key)` para autenticación de identidad.

El cliente debe enviar un mensaje cifrado al servidor y el servidor debe verificarlo y responder.


---
<a id='10'></a>
## 10. Referencias

### Estándares y RFCs
- **RFC 2104** — HMAC: Keyed-Hashing for Message Authentication. Bellare, Canetti, Krawczyk (1996). https://www.rfc-editor.org/rfc/rfc2104
- **RFC 7539 / RFC 8439** — ChaCha20 and Poly1305 for IETF Protocols. Nir, Langley (2018). https://www.rfc-editor.org/rfc/rfc8439
- **RFC 7465** — Prohibiting RC4 Cipher Suites. Popov (2015). https://www.rfc-editor.org/rfc/rfc7465
- **RFC 8452** — AES-GCM-SIV: Nonce Misuse-Resistant Authenticated Encryption (2019).
- **NIST SP 800-38B** — Recommendation for Block Cipher Modes of Operation: CMAC. https://doi.org/10.6028/NIST.SP.800-38B
- **NIST SP 800-38D** — Recommendation for Block Cipher Modes: GCM. https://doi.org/10.6028/NIST.SP.800-38D

### Libros y artículos académicos
- Boneh, D. & Shoup, V. (2023). *A Graduate Course in Applied Cryptography*, v0.6. https://crypto.stanford.edu/~dabo/cryptobook/
- Bernstein, D.J. (2005). *The Poly1305-AES message-authentication code*. FSE 2005.
- Bernstein, D.J. (2008). *ChaCha, a variant of Salsa20*. https://cr.yp.to/chacha.html
- Fluhrer, S., Mantin, I. & Shamir, A. (2001). *Weaknesses in the Key Scheduling Algorithm of RC4*. SAC 2001.
- McGrew, D. & Viega, J. (2004). *The Galois/Counter Mode of Operation (GCM)*. NIST submission.
- Krawczyk, H. (2001). *The Order of Encryption and Authentication for Protecting Communications*. CRYPTO 2001.
- Ferguson, N., Schneier, B. & Kohno, T. (2010). *Cryptography Engineering*. Wiley.
- Bellare, M., Pietrzak, K. & Rogaway, P. (2005). *Improved Security Analyses for CBC MACs*. CRYPTO 2005.

### Recursos online
- https://cryptopals.com — Retos prácticos (Sets 3 y 4 cubren stream ciphers y MACs)
- https://latacora.micro.blog/2018/04/03/cryptographic-right-answers.html — Guía de primitivas modernas
- https://soatok.blog — Blog técnico de criptografía práctica
- https://github.com/pyca/cryptography — Librería `cryptography` para Python
- https://pycryptodome.readthedocs.io — Librería PyCryptodome
- https://www.wireguard.com/papers/wireguard.pdf — WireGuard (usa ChaCha20-Poly1305)
